## Task

Your goal is to find out whether it is possible to reliably predict whether a website is a phishing site or not 
based on the 
easily obtainable information about the website. Based on the outcome, it may become possible to construct an 
automated system that warns users when they are about to visit a phishing website.

### Part 1: Decision tree

Your initial goal is to construct a small yet useful decision tree that predicts whether a website is a
phishing site or not.

The outcome should contain the following:
1. An image of the final decision tree.
2. Evaluation metrics for the decision tree.
3. Written instructions for an internet analyst to manually make the decision of whether the website
is likely to be a phishing site or not. The instructions must match one-to-one with your
decision tree, and be written in a way that is understandable to an engineer who is aware of the basics of internet technologies.

### Part 2: Random forest

As the ultimate goal is to build an automated system, you don't have to stick to a single, relatively simple decision tree.

Try to tweak the performance of the decision tree by replacing it with a random forest. You may also try to tune the 
hyperparameters of the random forest to improve the performance.

Be sure to include the validation results in your report.

> In real life, when you tune the hyperparameters based on the validation results, you should have yet another 
> data set that is not used for tuning the hyperparameters, but applied only once after the tuning of the 
> hyperparameters to obtain the final performance estimate of the tuned model.
> That is, there should be three sets: training, validation, and test sets. On this course, you may skip the need for the third set.

## CRISP-DM
This notebook follows the CRISP-DM process model

CRISP-DM phases:

1. Business understanding: The first phase is to understand the business problem that needs to be solved. What is the goal of the analysis? What are the requirements and constraints? What is the expected outcome?

2. Data understanding: The second phase is to collect and explore the data. What data is available? What are the characteristics of the data (variable types, value distributions etc.)? Are there any quality issues with the data (missing values, outliers, nonsensical values)?

3. Data preparation: The third phase is to preprocess the data. This includes cleaning the data, transforming the data, and selecting the relevant features. These steps should be documented in such detail that they can be reproduced later.

4. Modeling: The fourth phase is to choose a machine learning method and train the model. This phase also includes the validation of the model. Documentation needs include: which method was used, which parameters were used, what was the performance of the model?

5. Evaluation: The fifth phase is to evaluate the model. How well does the model perform? Does it meet the business requirements?

6. Deployment: The final phase is to deploy the model. How will the model be used in practice? How will the results be communicated? This phase may involve creating a recommendation of how to use the model in practice, or what to do next.


## 1. Business understanding

Goal of this project is to build a machine learning models to detect whether a website is a **Phishing site** or a **Legitimate  site** using easily obtainable website characteristics.

A **Phishing site** is marked as **-1** and a **Legitimate site** as **1**.

## 2. Data understanding

## 2.x Importing the data
The dataset contains website characteristics extracted from webpages, including URL structure, domain registration details, security certificate (SSL) features and page source attributes. The features are all easily obtainable without executing any risky code on the target websites.


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from pandas import DataFrame
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Fetch dataset
phishing_websites = fetch_ucirepo(id=327)

# Data
features : DataFrame = phishing_websites.data.features
targets : DataFrame = phishing_websites.data.targets

# Metadata and variables info
metadata : DataFrame = phishing_websites.metadata
variables : DataFrame = phishing_websites.variables

# Dynamically extract target column name to prevent KeyError
target_col = targets.columns[0]

print(f"Dataset Shape: {features.shape[0]} samples, {features.shape[1]} features")
print(f"\nClass Distribution in '{target_col}' (1 = Legitimate, -1 = Phishing):")
print(targets[target_col].value_counts())

Dataset Shape: 11055 samples, 30 features

Class Distribution in 'result' (1 = Legitimate, -1 = Phishing):
result
 1    6157
-1    4898
Name: count, dtype: int64


,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,rightclick,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,1,-1,-1,-1,-1,1,1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,1,-1,-1,0,-1,1,1,1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,1,-1,1,-1,1,0,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,1,-1,-1,1,-1,1,-1,1
4,1,0,-1,1,1,-1,1,1,-1,1,...,1,-1,1,-1,-1,0,-1,1,1,1


# 2.x Features & Metadata


In [4]:
# Example of data
features.head()

,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,rightclick,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,1,-1,-1,-1,-1,1,1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,1,-1,-1,0,-1,1,1,1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,1,-1,1,-1,1,0,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,1,-1,-1,1,-1,1,-1,1
4,1,0,-1,1,1,-1,1,1,-1,1,...,1,-1,1,-1,-1,0,-1,1,1,1


In [5]:
# Metadata
metadata


{'uci_id': 327,
 'name': 'Phishing Websites',
 'repository_url': 'https://archive.ics.uci.edu/dataset/327/phishing+websites',
 'data_url': 'https://archive.ics.uci.edu/static/public/327/data.csv',
 'abstract': 'This dataset collected mainly from: PhishTank archive, MillerSmiles archive, Googleâ€™s searching operators.',
 'area': 'Computer Science',
 'tasks': ['Classification'],
 'characteristics': ['Tabular'],
 'num_instances': 11055,
 'num_features': 30,
 'feature_types': ['Integer'],
 'demographics': [],
 'target_col': ['result'],
 'index_col': None,
 'has_missing_values': 'no',
 'missing_values_symbol': None,
 'year_of_dataset_creation': 2012,
 'last_updated': 'Tue Mar 05 2024',
 'dataset_doi': '10.24432/C51W2X',
 'creators': ['Rami Mohammad', 'Lee McCluskey'],
 'intro_paper': {'ID': 396,
  'type': 'NATIVE',
  'title': 'An assessment of features related to phishing websites using an automated technique',
  'authors': 'R. Mohammad, F. Thabtah, L. Mccluskey',
  'venue': 'International

### 2.6 Variables

The dataset should not have any missing variables.

In [6]:
# Variable information
variables

,name,role,type,demographic,description,units,missing_values
0,having_ip_address,Feature,Integer,None,None,None,no
1,url_length,Feature,Integer,None,None,None,no
2,shortining_service,Feature,Integer,None,None,None,no
3,having_at_symbol,Feature,Integer,None,None,None,no
4,double_slash_redirecting,Feature,Integer,None,None,None,no
5,prefix_suffix,Feature,Integer,None,None,None,no
6,having_sub_domain,Feature,Integer,None,None,None,no
7,sslfinal_state,Feature,Integer,None,None,None,no
8,domain_registration_length,Feature,Integer,None,None,None,no
9,favicon,Feature,Integer,None,None,None,no


### 2.7 Data Understanding: Summary

- The dataset contains 11 055 total website entries with 30 explanatory features as well as 1 target variable **Result**. This makes the total of 30 variables per entry.

- All 30 features are categorical taking values of **-1, 0 or 1**. These values indicate whether it's phishing, suspicious or legitimate.